# Dash Detector — Kaggle GPU notebook (Stages 2 & 3)

Pipeline overview:

| Stage | What | Where |
|-------|------|-------|
| **1** | process data — dash detection + CSV review | local |
| **2** | process data — CNN (ResNet18) feature extraction | **this notebook (GPU)** |
| **3** | train model — fixed-delay DNN (completion-spike target) | **this notebook (GPU)** |
| **4** | run model — inference on new video | local |

This notebook does Stage 2 then Stage 3. Stage 1 is already done locally (you
reviewed `dash_intervals.csv`); Stage 4 you run locally with `stage4.py`.

> **FDNN variant.** This is the fixed-delay net (`dash_code/dnn.py`,
> `FixedDelayLayer`): each layer applies fixed integer delays per channel-group
> with base-2 dilation across the stack — there are **no learned delays and no
> `--residual` / `--norm` knobs** (those were the SDSNN's). The checkpoint is
> still written as `checkpoints/sdsnn.pt`, which `stage4.py` loads.

**Before running**, set the GPU accelerator (Settings → Accelerator → GPU) and
attach your inputs as Kaggle datasets:
1. **dash-code** — the project's `dash_code/` modules. It's now ONE flat package
   (no `stage_1_2/`/`stage_3/` subfolders), so just upload every `.py`. The Setup
   cell copies them all into a `dash_code/` package — no per-file routing, so
   adding a module never needs a notebook change. **Re-upload after any code change.**
2. **dash-videos** — your gameplay videos **and** the reviewed `dash_intervals.csv`.
3. **dash-features** _(optional)_ — a previous run's `processed_features.zip`. Set
   `FEATURES_INPUT` to its path to **reuse** those features: Stage 2 is
   incremental, so it keeps the cached `.npz` and only extracts videos you've
   added since (and refreshes labels from the current CSV).

Edit the paths in the Setup cell to match where Kaggle mounted them.

## Setup — paths, env vars, GPU check

All scripts read their paths from environment variables (see `dash_code/config.py`),
so we only set env vars here. Edit `CODE_DIR` / `VIDEOS_DIR` to wherever Kaggle
mounted your datasets — they can be nested (e.g.
`/kaggle/input/datasets/<user>/dash-code`); the snippet in the cell prints the
real paths if you're unsure.

The code is one flat package now: scripts import `from dash_code import ...` and run
as `python -m dash_code.<module>`. This cell just copies every `.py` from the
dataset into a `dash_code/` package in `/kaggle/working/code` — flat dataset or one
already containing a `dash_code/` folder both work. Outputs go to `/kaggle/working`
so you can download them or save them as a dataset for reuse.

After running, check the prints: `dash_code` should list the `.py` modules (incl.
`dnn.py`) and `csv ... exists: True` (the reviewed labels must be in the videos dataset).

In [ ]:
import os, sys, shutil, glob
from pathlib import Path

# >>> EDIT THESE to match your attached datasets <<<
# Kaggle mounts datasets under /kaggle/input/<slug>; yours came out nested as
# /kaggle/input/datasets/adhitya1820/... — run the discovery snippet below if
# unsure (it prints the real paths):
#   import os
#   for dp, dn, fn in os.walk('/kaggle/input'):
#       if dp.count('/') - 2 <= 4: print(dp)
# NOTE: the FDNN code (with dnn.py / FixedDelayLayer) lives in dash-code-newnn.
# The older dash-code dataset is the SDSNN code (sdsnn.py) — do NOT use it here.
CODE_DIR   = '/kaggle/input/datasets/adhitya1820/dash-code-newnn'  # flat dash_code modules (FDNN: has dnn.py)
VIDEOS_DIR = '/kaggle/input/datasets/adhitya1820/dash-videos'  # .mp4/.mkv clips + the CSV
CSV_PATH   = f'{VIDEOS_DIR}/dash_intervals.csv'                 # reviewed labels (Stage 1)

# >>> FEATURES_INPUT: point this at a dataset holding cached .npz features (from a
# previous run's processed_features.zip). The .npz are found RECURSIVELY and copied
# into the working features dir. Stage 2 is INCREMENTAL, so these are REUSED and
# only new/changed videos get re-extracted (it no longer skips Stage 2 wholesale).
# Leave None to extract every video's features fresh in this run.
FEATURES_INPUT = '/kaggle/input/datasets/adhitya1820/dash-features'

WORK         = '/kaggle/working'
PROCESSED    = f'{WORK}/processed'
CHECKPOINTS  = f'{WORK}/checkpoints'

# reuse cached features: copy the dataset's .npz into the working features dir
# (search recursively so any nesting works). Stage 2 then reuses these and only
# decodes videos that are new or whose frame count changed.
if FEATURES_INPUT:
    found = glob.glob(os.path.join(FEATURES_INPUT, '**', '*.npz'), recursive=True)
    dst = os.path.join(PROCESSED, 'features')
    os.makedirs(dst, exist_ok=True)
    for p in found:
        shutil.copy2(p, dst)
    print(f'REUSING {len(found)} cached feature file(s) from {FEATURES_INPUT} '
          f'-> {dst}  (Stage 2 will extract only new/changed videos)')
    if not found:
        print('  WARNING: no .npz found under FEATURES_INPUT — check the path!')

# Copy the code into a writable dir as the `dash_code` package. The code is now a
# SINGLE flat package (no stage_1_2/stage_3 subfolders), so this is trivial: copy
# every *.py from the dataset into dash_code/ and ensure an __init__.py. No
# per-file routing, no hardcoded module list — add a module and it just works.
# The dataset can be flat (modules at top level) or already contain a dash_code/
# folder; both are handled.
CODE_RUN = f'{WORK}/code'
if os.path.exists(CODE_RUN):
    shutil.rmtree(CODE_RUN)
pkg = f'{CODE_RUN}/dash_code'
os.makedirs(pkg, exist_ok=True)

src = os.path.join(CODE_DIR, 'dash_code')
if not os.path.isdir(src):
    src = CODE_DIR                          # flat dataset: modules at top level
for f in glob.glob(os.path.join(src, '*.py')):
    shutil.copy2(f, pkg)
init = os.path.join(pkg, '__init__.py')
if not os.path.exists(init):
    open(init, 'w').close()                 # ensure the package is importable

# Paths consumed by dash_code/config.py  (features always read from working dir now)
os.environ['DASH_RAW_DIR']       = VIDEOS_DIR
os.environ['DASH_LABELS_CSV']    = CSV_PATH
os.environ['DASH_PROCESSED_DIR'] = PROCESSED      # Stage 3 reads features/ from here
os.environ['DASH_CKPT_DIR']      = CHECKPOINTS    # checkpoints always to working dir
# Leave DASH_BACKBONE unset -> default general-purpose ImageNet ResNet18.

print('dash_code:', sorted(os.listdir(pkg)))
print('videos   :', VIDEOS_DIR, '— exists:', os.path.isdir(VIDEOS_DIR))
print('csv      :', CSV_PATH, '— exists:', os.path.isfile(CSV_PATH))
feat_dir = os.path.join(PROCESSED, 'features')
n_npz = len(glob.glob(f'{feat_dir}/*.npz')) if os.path.isdir(feat_dir) else 0
print('features :', feat_dir, '—', n_npz, 'npz')

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu  :', torch.cuda.get_device_name(0))

## Stage 2 — process data (CNN feature extraction, incremental)

Runs the frozen ImageNet ResNet18 over the clips and saves per-video
`[n_frames, 512]` features + per-frame labels (from the reviewed CSV) to
`processed/features/*.npz`.

**Incremental:** it reuses any cached `.npz` already present (copied from
`FEATURES_INPUT` in Setup) and only **decodes videos that are new or whose frame
count changed** — every video's labels are still rebuilt from the current CSV,
so re-detected/edited labels are refreshed without re-decoding. Add new videos to
your dataset + CSV and re-run: only the new ones get extracted. `--force-features`
re-extracts everything.

**Decode on the GPU:** we install `torchcodec`, which `features.py` prefers — it
decodes on the GPU's hardware decoder (NVDEC) and keeps frames on-GPU through
ResNet, so decode isn't CPU-bound. It falls back to decord (CPU) then OpenCV if
torchcodec can't initialise; the decoder actually used is printed near the top of
the output (look for `decoder: torchcodec (GPU/NVDEC)`).

Download `/kaggle/working/processed` (or save it as a dataset) afterward so the
next run reuses these `.npz` and only extracts whatever you've added since.

In [ ]:
import subprocess

# Stage 2 is now INCREMENTAL (see dash_code/process_data.py). It reuses any cached
# .npz already copied into the working features dir (from FEATURES_INPUT) and
# only DECODES videos that are new or whose frame count changed, while rebuilding
# every video's labels (the soft Gaussian completion target) from the current CSV.
# So we ALWAYS run it — even when reusing features — to pick up newly added videos
# and refresh stale labels. (Use --force-features to re-extract everything.)
#
# MULTI-GPU: extraction auto-detects GPUs and splits the video list across them
# (one worker per GPU, round-robin). On a "GPU T4 x2" session this uses BOTH cards
# and ~halves a from-scratch extraction. Override with --gpus N (0=auto, 1=single).
#
# Decoder: install torchcodec so the new videos decode on the GPU's hardware
# decoder (NVDEC) — features.py prefers it, falling back to decord (CPU) then
# OpenCV if torchcodec can't initialise. decord is kept as that CPU fallback.
subprocess.run('pip -q install torchcodec decord', shell=True)

if FEATURES_INPUT:
    feat_dir = os.path.join(os.environ['DASH_PROCESSED_DIR'], 'features')
    n_cached = len(glob.glob(f'{feat_dir}/*.npz')) if os.path.isdir(feat_dir) else 0
    print(f'Incremental Stage 2: reusing {n_cached} cached .npz, extracting only '
          f'new/changed videos, refreshing labels from the CSV.')

subprocess.run(f'cd {CODE_RUN} && python -m dash_code.process_data --stage features',
               shell=True, check=True)

## Stage 3 — train model (fixed-delay DNN, completion-spike target)

Loads the `.npz` features, splits by video, and trains the fixed-delay DNN with
**heatmap regression** (weighted MSE) on the soft Gaussian **completion** target —
the model learns to emit a confidence bump that peaks when a dash has fully
unfolded, so #peaks == #dashes. Logs per-epoch val **count MAE + peak-timing
P/R/F1** and saves the best checkpoint to `checkpoints/sdsnn.pt` plus
`metrics.json`. (`--loss` is accepted for back-compat but routed to heatmap; the
target is soft, so plain BCE/focal don't apply.)

**Architecture experiments:** set `LAYERS` and `HIDDEN` in the cell below. To
probe depth, bump `LAYERS` — each layer adds a fixed delay step with the next
base-2 dilation (1, 2, 4, 8, …), so a deeper stack just widens the receptive
field. The `FixedDelayLayer` has **no `--residual` / `--norm` flags** (those were
the learned-delay SDSNN's; `train.py` here doesn't define them, so passing them
would error) — keep `EXTRA` to real knobs like `--lr` / `--epochs`. Each epoch
still prints **per-layer gradient norms** (`grad L1 ... Ln`) so you can watch the
early layers as you add depth. Change ONE knob at a time so results aren't
confounded — and remember the val set is tiny, so small moves are noise.

After training it runs an **NMS-threshold sweep** on the best model — for each
confidence cutoff it counts peaks (1-D non-max suppression) and reports count MAE
+ peak-timing P/R/F1. The best-timing-F1 threshold is baked into the checkpoint
(with the peak min-distance and match tolerance) and the full table goes to
`checkpoints/sweep.json`. Cheap, and reuses the cached features.

In [ ]:
# --- architecture knobs (change & re-run; cached features are reused) ---
LAYERS = 4          # number of fixed-delay layers (each adds the next base-2 dilation)
HIDDEN = 64         # delay-layer width (try 64 vs 48)
EXTRA  = ''         # FixedDelayLayer has NO --residual/--norm flags (would error here).
                    # Use real knobs instead, e.g. '--lr 5e-4', '--epochs 40'. (loss is heatmap)

!cd {CODE_RUN} && python -m dash_code.train --epochs 30 --layers {LAYERS} --hidden {HIDDEN} {EXTRA}

## Outputs (for Stage 4, local) + cache the features

This cell bundles everything for download from `/kaggle/working`:
- **`dash_outputs.zip`** — `sdsnn.pt` (bundles `in_dim`, `hidden`, `layers`,
  `img_size`, `backbone`, and the tuned NMS `threshold` + `peak_min_dist` /
  `peak_match_tol`), `metrics.json`, `sweep.json`.
- **`processed_features.zip`** — the per-video CNN features + labels. Upload this
  as a Kaggle dataset and set `FEATURES_INPUT` in the Setup cell next time to
  **skip Stage 2 entirely** (the GPU-heavy part) and go straight to training.

It also prints the trained checkpoint's metrics and the count/timing NMS sweep table.

In [ ]:
import json, os, shutil

# --- bundle outputs for download ---
shutil.make_archive(f'{WORK}/dash_outputs', 'zip', CHECKPOINTS)
print('outputs  -> dash_outputs.zip (sdsnn.pt + metrics.json + sweep.json)')

# package the cached feature vectors so you can skip Stage 2 next time
feat_dir = os.path.join(os.environ['DASH_PROCESSED_DIR'], 'features')
if not FEATURES_INPUT and os.path.isdir(feat_dir) and os.listdir(feat_dir):
    shutil.make_archive(f'{WORK}/processed_features', 'zip', os.environ['DASH_PROCESSED_DIR'])
    n = len([f for f in os.listdir(feat_dir) if f.endswith('.npz')])
    print(f'features -> processed_features.zip ({n} .npz). Upload it as a dataset '
          f'and set FEATURES_INPUT next time to skip Stage 2.')

# --- read back the metrics ---
ckpt = f'{CHECKPOINTS}/sdsnn.pt'
print('\ncheckpoint exists:', os.path.isfile(ckpt))

mpath = f'{CHECKPOINTS}/metrics.json'
if os.path.isfile(mpath):
    hist = json.load(open(mpath))
    print('epochs trained:', len(hist))
    print('last epoch    :', hist[-1])

# completion-count metrics + the NMS-threshold sweep (the numbers that matter):
# count MAE = mean |pred_count - gt_count|; timing F1 = peaks matched to GT
# completions within +/- match-tol frames. The best-F1 threshold is baked in.
spath = f'{CHECKPOINTS}/sweep.json'
if os.path.isfile(spath):
    sw = json.load(open(spath))
    b = sw['best']
    print(f"\nbest timing F1={b['f1']:.3f}  P={b['precision']:.2f}  R={b['recall']:.2f}  "
          f"count MAE={b['count_mae']:.3f}  @ threshold={b['threshold']:.2f}  "
          f"(tp/fp/fn={b['tp']}/{b['fp']}/{b['fn']})")
    print("\n  thr | count MAE | timing P/R/F1")
    for r in sw['sweep']:
        print(f"  {r['threshold']:.2f} | {r['count_mae']:>8.3f}  | "
              f"{r['precision']:.2f}/{r['recall']:.2f}/{r['f1']:.2f}")

In [ ]:
# --- everything in ONE zip for a single download ---
import os, zipfile

bundle = f'{WORK}/dash_allv2.zip'
roots = [CHECKPOINTS, os.path.join(os.environ['DASH_PROCESSED_DIR'], 'features')]

with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                full = os.path.join(dirpath, fn)
                # store as checkpoints/... or features/... inside the zip
                arc = os.path.join(os.path.basename(root.rstrip('/')),
                                   os.path.relpath(full, root))
                z.write(full, arc)

print('bundled -> dash_allv2.zip')
with zipfile.ZipFile(bundle) as z:
    for i in z.infolist():
        print(f'  {i.file_size:>10}  {i.filename}')